# Eval v2: frozen baseline, validation-based selection, GeM ablation, upscaled multi-scale (no training)

Evaluation-only run (paper v2). It reuses the trained B0-B3 checkpoints, attached as inputs from
the training kernels (`ckpt_last.pt`, `whiten_<V>-finetune.pkl`, `vecs_<V>-finetune_<ds>.pt`).

| Group | What | Why |
|---|---|---|
| `frozen` | DINOv2 ViT-B/14 without fine-tuning, same protocol (GeM/CLS/Avg, SS/MS, with and without Lw, with and without alpha-QE) | fair comparison: no-QE vs no-QE, QE vs QE |
| `validation` | retrieval mAP + contrastive loss on the rSfM-120k **val** split (3D models disjoint from train and test) for frozen, B0-B3 | select the version on validation, not on ROxford |
| `validation_qe` | alpha-QE grid (alpha in {0,1,3,5,10}, n in {1,2,3,5,10}) on val | choose alpha, n on validation |
| `test_all` | B0-B3 re-scored on ROxf/RPar from their saved descriptors with the val-chosen alpha-QE | final report only |
| `ablation_gem` | GeM with ReLU vs shifted GeM (x - min) vs signed GeM, on frozen and B0 | test the claim that ReLU hurts GeM on ViT tokens |
| `B6_upscale` | B3 CLS with scales [sqrt2, 1, 1/sqrt2] vs [1, 1/sqrt2, 1/2] | test why multi-scale hurts the class token |

Settings: GPU T4, Internet on. Inputs: dataset `rofiqsamanhudi/rparis6k-and-roxford5k` + the 4 training kernels.
The code below is the repository package `src/` (written to disk, then run as `python -m src.eval_v2`).

In [ ]:
import os
os.makedirs('src', exist_ok=True)
print(sorted(os.listdir('/kaggle/input')))

In [ ]:
%%writefile src/__init__.py
"""CBIR landmark retrieval: DINOv2 + GeM / CLS, contrastive fine-tuning, Lw whitening, alpha-QE."""


In [ ]:
%%writefile src/config.py
"""Default configuration and per-version presets (B0..B4)."""
import math

DEFAULT = dict(
    data_root='/tmp/cbir_data',      # datasets are downloaded here
    out_dir='outputs/b3',            # results + checkpoints
    arch='dinov2_vitb14',
    datasets=['roxford5k', 'rparis6k'],
    test_imsize=518,                 # longest side, rounded down to a multiple of 14
    multiscale=[1, 1 / 2 ** 0.5, 1 / 2],
    gem_p=3.0,
    train_head='cls',                # descriptor used for loss + mining: 'gem' or 'cls'
    whiten_max_pairs=5000,           # q/p pairs used to learn Lw
    # ---- training ----
    train_imsize=518,
    finetune_blocks=2,               # number of last ViT blocks to train (ViT-B has 12)
    epochs=10,
    lr=1e-6,
    p_lr_mult=10,                    # lr of GeM p = lr * 10
    weight_decay=1e-6,
    lr_gamma=math.exp(-0.01),        # ExponentialLR
    margin=0.85,                     # contrastive loss margin
    query_size=2000,
    pool_size=22000,
    neg_num=5,
    batch_size=5,                    # tuples per optimizer step
    max_train_hours=8.0,
    eval_every_epoch=False,
    resume_path='',
    num_workers=4,
    amp=True,                        # fp16 autocast on CUDA
    seed=0,
    # ---- test-time ----
    qe=False,                        # alpha-QE rows (B4)
    qe_alpha=3,
    qe_n=2,
)

# Each version changes one factor vs the best previous one.
PRESETS = {
    'b0': dict(train_head='gem', finetune_blocks=2, lr=2e-6),
    'b1': dict(train_head='cls', finetune_blocks=2, lr=2e-6),
    'b2': dict(train_head='cls', finetune_blocks=4, lr=2e-6),
    'b3': dict(train_head='cls', finetune_blocks=2, lr=1e-6),
    'b4': dict(train_head='cls', finetune_blocks=2, lr=1e-6, qe=True),  # B3 model + alpha-QE
}


def get_config(version='b3', **overrides):
    cfg = dict(DEFAULT)
    cfg.update(PRESETS[version])
    cfg['out_dir'] = f'outputs/{version}'
    cfg.update({k: v for k, v in overrides.items() if v is not None})
    return cfg


In [ ]:
%%writefile src/data.py
"""Dataset download (ROxford5k / RParis6k / rSfM-120k), image loading and training tuples."""
import glob
import os
import pickle
import shutil
import subprocess

import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # ImageNet normalization (DINOv2)
PATCH = 14
CMP = 'http://cmp.felk.cvut.cz/cnnimageretrieval/data'


# ---------------------------------------------------------------- download
def sh(cmd):
    print('$', cmd)
    subprocess.run(cmd, shell=True, check=True)


def disk(path):
    u = shutil.disk_usage(path)
    return f'{path}: free {u.free / 2**30:.1f} GB / total {u.total / 2**30:.1f} GB'


def remote_size_gb(url):
    try:
        out = subprocess.run(f'curl -sIL --max-time 30 "{url}"', shell=True, capture_output=True, text=True).stdout
        sizes = [int(l.split(':')[1]) for l in out.splitlines() if l.lower().startswith('content-length')]
        return sizes[-1] / 2**30 if sizes else None
    except Exception:
        return None


def fetch(url, dst):
    if os.path.exists(dst):
        return
    try:
        sh(f'wget -q --tries=3 --timeout=60 -O "{dst}.part" "{url}" && mv "{dst}.part" "{dst}"')
    except subprocess.CalledProcessError:
        if os.path.exists(dst + '.part'):
            os.remove(dst + '.part')
        raise RuntimeError(f'Download failed: {url}')


def use_kaggle_input(root):
    """Use the Kaggle dataset rofiqsamanhudi/rparis6k-and-roxford5k when it is attached."""
    found = {}
    for ds in ['roxford5k', 'rparis6k']:
        g = sorted(glob.glob(f'/kaggle/input/**/{ds}/gnd_{ds}.pkl', recursive=True))
        if not (g and os.path.isdir(os.path.join(os.path.dirname(g[0]), 'jpg'))):
            continue
        src, d = os.path.dirname(g[0]), f'{root}/test/{ds}'
        os.makedirs(d, exist_ok=True)
        if not os.path.exists(f'{d}/jpg'):
            # rparis6k is nested (jpg/paris/<landmark>/*.jpg) -> flat folder of symlinks
            os.makedirs(f'{d}/jpg')
            for fp in glob.glob(f'{src}/jpg/**/*.jpg', recursive=True):
                dstf = f'{d}/jpg/{os.path.basename(fp)}'
                if not os.path.exists(dstf):
                    os.symlink(fp, dstf)
        if not os.path.exists(f'{d}/gnd_{ds}.pkl'):
            shutil.copy(g[0], f'{d}/gnd_{ds}.pkl')
        gl = pickle.load(open(f'{d}/gnd_{ds}.pkl', 'rb'))
        miss = [n for n in gl['imlist'] + gl['qimlist'] if not os.path.exists(f'{d}/jpg/{n}.jpg')]
        if miss:
            print(f'{ds}: {len(miss)} images missing from Kaggle input, downloading from the web')
            shutil.rmtree(f'{d}/jpg')
            continue
        found[ds] = src
        print(f'{ds}: using Kaggle input {src} ({len(os.listdir(f"{d}/jpg"))} images)')
    return found


def download_test(root):
    """Same as cirtorch/utils/download.py (download_test)."""
    found = use_kaggle_input(root)
    if len(found) == 2:
        return
    src = {'oxford5k': ('https://www.robots.ox.ac.uk/~vgg/data/oxbuildings', ['oxbuild_images-v1.tgz']),
           'paris6k': ('https://www.robots.ox.ac.uk/~vgg/data/parisbuildings', ['paris_1-v1.tgz', 'paris_2-v1.tgz'])}
    for ds, (base, files) in src.items():
        if ('r' + ds) in found:
            continue
        jpg = f'{root}/test/{ds}/jpg'
        if os.path.isdir(jpg) and len(os.listdir(jpg)) > 1000:
            continue
        tmp = f'{root}/test/{ds}/tmp'
        os.makedirs(tmp, exist_ok=True)
        os.makedirs(jpg, exist_ok=True)
        for f in files:
            fetch(f'{base}/{f}', f'{tmp}/{f}')
            sh(f'tar -xzf "{tmp}/{f}" -C "{tmp}" && rm "{tmp}/{f}"')
        sh(f'find "{tmp}" -type f -name "*.jpg" -exec mv -t "{jpg}" {{}} +')
        shutil.rmtree(tmp)
    for ds, base in [('roxford5k', 'oxford5k'), ('rparis6k', 'paris6k')]:
        if ds in found:
            continue
        d = f'{root}/test/{ds}'
        os.makedirs(d, exist_ok=True)
        if not os.path.exists(f'{d}/jpg'):
            os.symlink(f'{root}/test/{base}/jpg', f'{d}/jpg')
        fetch(f'{CMP}/test/{ds}/gnd_{ds}.pkl', f'{d}/gnd_{ds}.pkl')


def download_train(root):
    """Same as cirtorch download_train, but streams the tar to avoid using double disk space."""
    d = f'{root}/train/retrieval-SfM-120k'
    ims = f'{d}/ims'
    os.makedirs(ims, exist_ok=True)
    url = f'{CMP}/train/ims/ims.tar.gz'
    if not os.path.exists(f'{ims}/.done'):
        gb, free = remote_size_gb(url), shutil.disk_usage(root).free / 2**30
        print(f'ims.tar.gz ≈ {gb if gb is None else round(gb, 1)} GB, free {free:.1f} GB')
        if gb is not None and gb * 1.05 > free:
            raise RuntimeError('Not enough disk for rSfM-120k; change --data-root')
        sh(f'curl -L --retry 5 -sS "{url}" | tar -xzf - -C "{ims}" && touch "{ims}/.done"')
    for f in ['retrieval-SfM-120k.pkl', 'retrieval-SfM-120k-whiten.pkl']:
        fetch(f'{CMP}/train/dbs/{f}', f'{d}/{f}')
    print(disk(root))


def load_gnd(root, ds):
    with open(f'{root}/test/{ds}/gnd_{ds}.pkl', 'rb') as f:
        return pickle.load(f)


def cid2filename(cid, prefix):
    return os.path.join(prefix, cid[-2:], cid[-4:-2], cid[-6:-4], cid)


# ---------------------------------------------------------------- images
def load_img(path, imsize, bbx=None):
    img = Image.open(path).convert('RGB')
    if bbx is not None:
        img = img.crop(tuple(bbx))                       # test queries are cropped to their bbox
    img.thumbnail((imsize, imsize), Image.LANCZOS)        # downscale only, keep aspect ratio (as cirtorch)
    w, h = img.size
    w14, h14 = max(PATCH, w // PATCH * PATCH), max(PATCH, h // PATCH * PATCH)
    if (w14, h14) != (w, h):
        if w < PATCH or h < PATCH:
            img = img.resize((w14, h14), Image.BICUBIC)
        else:                                            # center-crop the < 14 px remainder
            l, t = (w - w14) // 2, (h - h14) // 2
            img = img.crop((l, t, l + w14, t + h14))
    x = torch.from_numpy(np.asarray(img, dtype=np.float32) / 255.).permute(2, 0, 1)
    return (x - torch.tensor(MEAN)[:, None, None]) / torch.tensor(STD)[:, None, None]


class ImageList(Dataset):
    def __init__(self, paths, imsize, bbxs=None):
        self.paths, self.imsize, self.bbxs = paths, imsize, bbxs

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        return load_img(self.paths[i], self.imsize, None if self.bbxs is None else self.bbxs[i])


# ---------------------------------------------------------------- training tuples
class Tuples:
    """Compact port of cirtorch TuplesDataset: (query, positive, hard negatives) from rSfM-120k."""

    def __init__(self, cfg, split='train'):
        self.cfg = cfg
        d = f"{cfg['data_root']}/train/retrieval-SfM-120k"
        db = pickle.load(open(f'{d}/retrieval-SfM-120k.pkl', 'rb'))[split]
        self.images = [cid2filename(c, f'{d}/ims') for c in db['cids']]
        self.clusters, self.qpool, self.ppool = db['cluster'], db['qidxs'], db['pidxs']
        print(f'{split}: {len(self.images)} images, {len(self.qpool)} q/p pairs')

    def mine(self, net, extract_fn):
        """Pick random queries and mine hard negatives (one per cluster). Returns mean negative distance."""
        cfg = self.cfg
        sel = torch.randperm(len(self.qpool))[:cfg['query_size']]
        self.qidxs = [self.qpool[i] for i in sel]
        self.pidxs = [self.ppool[i] for i in sel]
        pool = torch.randperm(len(self.images))[:cfg['pool_size']].tolist()
        qv = extract_fn(net, [self.images[i] for i in self.qidxs], cfg['train_imsize'], desc='mine q')[1]
        pv = extract_fn(net, [self.images[i] for i in pool], cfg['train_imsize'], desc='mine pool')[1]
        ranks = torch.argsort(pv.t() @ qv, dim=0, descending=True)
        self.nidxs, dsum = [], 0.
        for q in range(len(self.qidxs)):
            clusters, nid, r = {self.clusters[self.qidxs[q]]}, [], 0
            while len(nid) < cfg['neg_num'] and r < len(pool):
                cand = pool[ranks[r, q]]
                if self.clusters[cand] not in clusters:
                    nid.append(cand)
                    clusters.add(self.clusters[cand])
                    dsum += (qv[:, q] - pv[:, ranks[r, q]]).norm().item()
                r += 1
            self.nidxs.append(nid)
        return dsum / max(1, sum(len(n) for n in self.nidxs))


class TupleSet(Dataset):
    def __init__(self, tuples):
        self.T = tuples

    def __len__(self):
        return len(self.T.qidxs)

    def __getitem__(self, i):
        idx = [self.T.qidxs[i], self.T.pidxs[i]] + self.T.nidxs[i]
        return [load_img(self.T.images[j], self.T.cfg['train_imsize']) for j in idx]


In [ ]:
%%writefile src/evaluate.py
"""Revisited Oxford/Paris evaluation (Easy / Medium / Hard), ported from cirtorch."""
import numpy as np


def compute_ap(ranks, nres):
    ap, step = 0., 1. / nres
    for j, r in enumerate(ranks):
        p0 = 1. if r == 0 else j / r
        p1 = (j + 1) / (r + 1)
        ap += (p0 + p1) * step / 2.
    return ap


def compute_map(ranks, gnd, kappas=(1, 5, 10)):
    nq = len(gnd)
    aps, prs, nempty = np.zeros(nq), np.zeros((nq, len(kappas))), 0
    for i in range(nq):
        qgnd = np.array(gnd[i]['ok'])
        if qgnd.shape[0] == 0:
            aps[i] = np.nan
            prs[i] = np.nan
            nempty += 1
            continue
        qgndj = np.array(gnd[i].get('junk', []))
        pos = np.arange(ranks.shape[0])[np.isin(ranks[:, i], qgnd)]
        junk = np.arange(ranks.shape[0])[np.isin(ranks[:, i], qgndj)]
        k = ij = 0
        if len(junk):
            for ip in range(len(pos)):
                while ij < len(junk) and pos[ip] > junk[ij]:
                    k += 1
                    ij += 1
                pos[ip] -= k
        aps[i] = compute_ap(pos, len(qgnd))
        pos = pos + 1
        for j, kk in enumerate(kappas):
            kq = min(max(pos), kk)
            prs[i, j] = (pos <= kq).sum() / kq
    n = nq - nempty
    return np.nansum(aps) / n, np.nansum(prs, axis=0) / n


def eval_ranks(ranks, gnd):
    """ranks: Ndb x Nq. Returns {'E'|'M'|'H': {'mAP', 'mP@1', 'mP@5', 'mP@10'}} in %."""
    out = {}
    for name, ok, jk in [('E', ['easy'], ['junk', 'hard']),
                         ('M', ['easy', 'hard'], ['junk']),
                         ('H', ['hard'], ['junk', 'easy'])]:
        g = [{'ok': np.concatenate([q[k] for k in ok]), 'junk': np.concatenate([q[k] for k in jk])} for q in gnd]
        m, pr = compute_map(ranks, g)
        out[name] = dict(mAP=round(100 * m, 2), **{f'mP@{k}': round(100 * v, 2) for k, v in zip((1, 5, 10), pr)})
    return out


In [ ]:
%%writefile src/extract.py
"""Descriptor extraction (single- and multi-scale)."""
import gc

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from .data import PATCH, ImageList
from .model import HEADS


def resize14(img, s):
    H, W = img.shape[-2:]
    nh, nw = max(PATCH, round(H * s / PATCH) * PATCH), max(PATCH, round(W * s / PATCH) * PATCH)
    return F.interpolate(img, size=(nh, nw), mode='bilinear', align_corners=False, antialias=True)


@torch.no_grad()
def extract(net, paths, imsize, scales=(1,), bbxs=None, desc='extract', heads=False, num_workers=4, head_list=None):
    """Return {scale: D x N} for the train head, or {head: {scale: D x N}} if heads=True.

    head_list: which heads to keep (default: HEADS)."""
    head_list = head_list or HEADS
    net.eval()
    device = next(net.parameters()).device
    dl = DataLoader(ImageList(paths, imsize, bbxs), batch_size=1, shuffle=False, num_workers=num_workers)
    out = {h: {s: torch.zeros(net.dim, len(paths)) for s in scales} for h in head_list}
    for i, img in enumerate(tqdm(dl, desc=desc, mininterval=10)):
        img = img.to(device, non_blocking=True)
        for s in scales:
            o = net(img if s == 1 else resize14(img, s), heads=True)
            for h in head_list:
                out[h][s][:, i] = o[h].squeeze(0).cpu()
    del dl
    gc.collect()
    return out if heads else out[net.train_head]


def combine_ms(per_scale, p):
    """cirtorch extract_ms: l2n(mean_s(v_s^p)^(1/p)); p=1 -> plain mean (CLS/avg can be negative)."""
    v = torch.stack([per_scale[s] for s in per_scale])
    v = v.mean(0) if p == 1 else v.clamp(min=0).pow(p).mean(0).pow(1. / p)
    return F.normalize(v, dim=0)


def variants(per_scale, p):
    return {'ss': per_scale[1], 'ms': combine_ms(per_scale, p)}


def ms_power(head, p):
    """Multi-scale pooling power: GeM-style for non-negative GeM heads, plain mean otherwise."""
    return p if head in ('gem', 'gem_shift') else 1


In [ ]:
%%writefile src/model.py
"""DINOv2 backbone + GeM pooling; returns GeM, CLS and average-pooled descriptors."""
import torch
import torch.nn as nn
import torch.nn.functional as F

from .data import PATCH

HEADS = ['gem', 'cls', 'avg']                       # heads used by the training runs (B0-B4)
ABLATION_HEADS = ['gem_shift', 'gem_signed']        # GeM variants without ReLU (paper v2, Sec. ablation)
ALL_HEADS = HEADS + ABLATION_HEADS


class GeM(nn.Module):
    """Generalized-mean pooling with learnable p (Radenović et al., TPAMI 2018)."""

    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):  # B x D x h x w -> B x D
        return F.avg_pool2d(x.clamp(min=self.eps).pow(self.p), x.shape[-2:]).pow(1. / self.p).flatten(1)

    def signed(self, x):  # sign-preserving power mean: no ReLU, negative activations kept
        m = (x.sign() * x.abs().pow(self.p)).mean(dim=(2, 3))
        return m.sign() * m.abs().clamp(min=self.eps).pow(1. / self.p)


def flat_blocks(bb):
    out = []
    for b in bb.blocks:  # also supports BlockChunk
        out.extend([m for m in b if not isinstance(m, nn.Identity)] if isinstance(b, nn.ModuleList) else [b])
    return out


class RetrievalNet(nn.Module):
    """DINOv2 -> {gem: ReLU+GeM on patch tokens, cls: CLS token, avg: mean of patch tokens}, all L2-normalized.

    Only the last `finetune_blocks` blocks (+ final norm + GeM p) are trainable.
    """

    def __init__(self, backbone, p=3.0, finetune_blocks=0, train_head='cls', amp=True):
        super().__init__()
        self.backbone, self.pool = backbone, GeM(p)
        self.blocks = flat_blocks(backbone)
        self.n_frozen = len(self.blocks) - finetune_blocks
        self.train_head, self.amp = train_head, amp
        for prm in backbone.parameters():
            prm.requires_grad = False
        for b in self.blocks[self.n_frozen:]:
            for prm in b.parameters():
                prm.requires_grad = True
        if finetune_blocks > 0:
            for prm in backbone.norm.parameters():
                prm.requires_grad = True
        self.dim = backbone.embed_dim

    def forward(self, x, heads=False):
        bb = self.backbone
        B, _, H, W = x.shape
        h, w = H // PATCH, W // PATCH
        with torch.autocast('cuda', dtype=torch.float16, enabled=self.amp and x.is_cuda):
            with torch.no_grad():                               # frozen part: no graph stored
                t = bb.prepare_tokens_with_masks(x)
                for blk in self.blocks[:self.n_frozen]:
                    t = blk(t)
            for blk in self.blocks[self.n_frozen:]:
                t = blk(t)
            t = bb.norm(t)
        nreg = getattr(bb, 'num_register_tokens', 0)
        pt = t[:, 1 + nreg:].float()                           # B x N x D patch tokens
        fmap = pt.transpose(1, 2).reshape(B, -1, h, w)
        o = {'gem': F.normalize(self.pool(F.relu(fmap)), dim=-1),   # ReLU: ViT tokens can be negative
             'cls': F.normalize(t[:, 0].float(), dim=-1),
             'avg': F.normalize(pt.mean(1), dim=-1),
             # ablations (eval only): GeM after shifting each channel to be non-negative (x - min), and signed GeM
             'gem_shift': F.normalize(self.pool(fmap - fmap.amin(dim=(2, 3), keepdim=True)), dim=-1),
             'gem_signed': F.normalize(self.pool.signed(fmap), dim=-1)}
        return o if heads else o[self.train_head]


def load_backbone(arch='dinov2_vitb14', device='cpu'):
    return torch.hub.load('facebookresearch/dinov2', arch).to(device).eval()


def build_model(cfg, device, finetune_blocks=None, backbone=None):
    bb = backbone if backbone is not None else load_backbone(cfg['arch'], device)
    fb = cfg['finetune_blocks'] if finetune_blocks is None else finetune_blocks
    return RetrievalNet(bb, p=cfg['gem_p'], finetune_blocks=fb,
                        train_head=cfg['train_head'], amp=cfg['amp']).to(device)


# Default location of trained weights (not in the repository; available on request).
# B4 = B3 weights + alpha-QE at test time.
WEIGHTS = {v: (f'weights/{v.upper()}/ckpt_last.pt', f'weights/{v.upper()}/whiten.pkl')
           for v in ['b0', 'b1', 'b2', 'b3']}
WEIGHTS['b4'] = WEIGHTS['b3']


def load_weights(cfg, device, path):
    """Build the model and load a fine-tuned checkpoint (only the trained blocks + norm + GeM p are stored)."""
    net = build_model(cfg, device)
    s = torch.load(path, map_location=device)
    r = net.load_state_dict(s.get('trainable', s), strict=False)
    if r.unexpected_keys:
        raise ValueError(f'{path} does not match finetune_blocks={cfg["finetune_blocks"]}: {r.unexpected_keys[:3]}')
    print(f'loaded weights from {path} (epoch {s.get("epoch", "?")})')
    return net.eval()


In [ ]:
%%writefile src/qe.py
"""alpha-weighted query expansion (Radenović et al., TPAMI 2018, Sec. 4)."""
import numpy as np


def aqe(X, Q, n=2, alpha=3):
    """q' = l2n(q + sum_{i<=n} sim_i^alpha * x_i). X: D x Ndb, Q: D x Nq (L2-normalized)."""
    S = X.T @ Q
    top = np.argsort(-S, axis=0)[:n]
    Qn = Q.copy()
    for j in range(Q.shape[1]):
        w = np.clip(S[top[:, j], j], 0, None) ** alpha
        Qn[:, j] = Q[:, j] + (X[:, top[:, j]] * w).sum(1)
    return Qn / (np.linalg.norm(Qn, axis=0, keepdims=True) + 1e-6)


In [ ]:
%%writefile src/whiten.py
"""Learned whitening Lw (Radenović et al.), ported from cirtorch/utils/whiten.py."""
import numpy as np


def cholesky(S):
    alpha = 0
    while True:
        try:
            return np.linalg.cholesky(S + alpha * np.eye(*S.shape))
        except np.linalg.LinAlgError:
            alpha = 1e-10 if alpha == 0 else alpha * 10


def whitenlearn(X, qidxs, pidxs):
    """X: D x N descriptors; (qidxs[i], pidxs[i]) are matching pairs. Returns (m, P)."""
    m = X[:, qidxs].mean(axis=1, keepdims=True)
    df = X[:, qidxs] - X[:, pidxs]
    S = df @ df.T / df.shape[1]
    P = np.linalg.inv(cholesky(S))
    df = P @ (X - m)
    eigval, eigvec = np.linalg.eigh(df @ df.T)
    order = eigval.argsort()[::-1]
    return m, eigvec[:, order].T @ P


def whitenapply(X, m, P):
    X = P @ (X - m)
    return X / (np.linalg.norm(X, axis=0, keepdims=True) + 1e-6)


In [ ]:
%%writefile src/eval_v2.py
"""Evaluation-only experiments (paper v2). No training.

  frozen        DINOv2 ViT-B/14 without fine-tuning, same protocol as B0-B4 (all heads, SS/MS, +-Lw, +-alpha-QE)
  validation    model selection on the rSfM-120k *val* split (frozen, B0-B3): retrieval mAP + contrastive loss,
                and the alpha-QE (alpha, n) grid, so that ROxford/RParis are used only for the final report
  ablation_gem  GeM without ReLU on frozen and B0: shifted GeM (x - min) and signed GeM
  B6_upscale    B3 CLS descriptor with the upscaled scale set [sqrt2, 1, 1/sqrt2]

Inputs: the outputs of the B0-B3 training kernels (ckpt_last.pt, whiten_<V>-finetune.pkl, vecs_<V>-finetune_<ds>.pt),
found anywhere below --ckpt-root (on Kaggle: /kaggle/input).

    python -m src.eval_v2 --ckpt-root /kaggle/input --out-dir outputs/eval_v2
"""
import argparse
import glob
import json
import os
import pickle
import random
import time

import numpy as np
import pandas as pd
import torch

from .config import get_config
from .data import cid2filename, disk, download_test, download_train, load_gnd
from .evaluate import compute_map, eval_ranks
from .extract import combine_ms, extract, ms_power, variants
from .model import ALL_HEADS, build_model, load_backbone
from .qe import aqe
from .whiten import whitenapply, whitenlearn

VERSIONS = ['B0', 'B1', 'B2', 'B3']
FINETUNE_BLOCKS = {'frozen': 0, 'B0': 2, 'B1': 2, 'B2': 4, 'B3': 2}
TRAINED_HEAD = {'frozen': 'cls', 'B0': 'gem', 'B1': 'cls', 'B2': 'cls', 'B3': 'cls'}
QE_ALPHAS = [0, 1, 3, 5, 10]
QE_NS = [1, 2, 3, 5, 10]
SQ2 = 2 ** 0.5
UP_SCALES = [SQ2, 1, 1 / SQ2]


class Collector:
    """Rows grouped by experiment; each group is written to <out_dir>/<group>/results.{csv,json}."""

    def __init__(self, out_dir):
        self.out_dir, self.rows = out_dir, {}

    def add(self, group, **row):
        self.rows.setdefault(group, []).append(row)
        print(group, {k: v for k, v in row.items() if k not in ('group',)})

    def add_test(self, group, stage, ds, head, scale, lw, qe, r):
        self.add(group, stage=stage, dataset=ds, head=head, scale=scale, Lw=lw, qe=qe,
                 **{f'{k}_{m}': v for k, d in r.items() for m, v in d.items()})

    def save(self):
        for g, rows in self.rows.items():
            d = f'{self.out_dir}/{g}'
            os.makedirs(d, exist_ok=True)
            json.dump(rows, open(f'{d}/results.json', 'w'), indent=1, default=lambda o: o.item() if hasattr(o, 'item') else str(o))
            pd.DataFrame(rows).to_csv(f'{d}/results.csv', index=False)


def find_inputs(root):
    """Locate ckpt / whitening / saved test descriptors of B0-B3 below root."""
    found = {}
    for v in VERSIONS:
        w = sorted(glob.glob(f'{root}/**/whiten_{v}-finetune.pkl', recursive=True))
        if not w:
            raise FileNotFoundError(f'whiten_{v}-finetune.pkl not found below {root}')
        d = os.path.dirname(w[0])
        found[v] = dict(ckpt=f'{d}/ckpt_last.pt', whiten=w[0],
                        vecs={ds: f'{d}/vecs_{v}-finetune_{ds}.pt' for ds in ['roxford5k', 'rparis6k']})
        print(v, '->', d, sorted(os.listdir(d)))
    return found


def get_scale(per_scale, s):
    """Saved descriptor dicts use float scale keys; match them approximately."""
    for k in per_scale:
        if abs(float(k) - s) < 1e-6:
            return per_scale[k]
    raise KeyError(s)


def make_net(cfg, device, v, ckpt=None):
    net = build_model(cfg, device, finetune_blocks=FINETUNE_BLOCKS[v])
    if ckpt:
        s = torch.load(ckpt, map_location=device)
        r = net.load_state_dict(s.get('trainable', s), strict=False)
        assert not r.unexpected_keys, r.unexpected_keys[:3]
        print(f'{v}: loaded {ckpt} (epoch {s.get("epoch")}), GeM p = {net.pool.p.item():.3f}')
    return net.eval()


# ------------------------------------------------------------------ validation split
def build_val(cfg, nq, ndb, seed=0):
    """Fixed retrieval task on the rSfM-120k val split (3D models disjoint from train and from Oxford/Paris).

    Queries: nq unique val query images. Database: their positives + random val images (queries excluded),
    ndb images in total. Relevant = database images from the same 3D model (cluster) as the query.
    """
    d = f"{cfg['data_root']}/train/retrieval-SfM-120k"
    db = pickle.load(open(f'{d}/retrieval-SfM-120k.pkl', 'rb'))['val']
    rs = np.random.RandomState(seed)
    pairs, seen = [], set()
    for i in rs.permutation(len(db['qidxs'])):
        q = db['qidxs'][i]
        if q not in seen:
            seen.add(q)
            pairs.append((q, db['pidxs'][i]))
        if len(pairs) == nq:
            break
    qset = {q for q, _ in pairs}
    dbidx = list(dict.fromkeys(p for _, p in pairs if p not in qset))
    have = set(dbidx)
    for i in rs.permutation(len(db['cids'])):
        if len(dbidx) >= ndb:
            break
        if i not in qset and i not in have:
            dbidx.append(int(i))
            have.add(i)
    clusters = np.asarray(db['cluster'])
    qidx = [q for q, _ in pairs]
    pos_in_db = {p: j for j, p in enumerate(dbidx)}
    gnd = [{'ok': np.where(clusters[dbidx] == clusters[q])[0], 'junk': np.array([], dtype=int)} for q in qidx]
    paths = lambda ids: [cid2filename(db['cids'][i], f'{d}/ims') for i in ids]
    print(f'val task: {len(qidx)} queries, {len(dbidx)} database images, '
          f'{np.mean([len(g["ok"]) for g in gnd]):.1f} relevant / query')
    return dict(qpaths=paths(qidx), dbpaths=paths(dbidx), gnd=gnd,
                pos=[pos_in_db.get(p, -1) for _, p in pairs],
                qcl=clusters[qidx], dbcl=clusters[dbidx])


def mine_val_negatives(val, X, Q, n_neg=5):
    """Fixed hard negatives (one per other 3D model), mined once with the frozen model, reused for every model."""
    S = X.T @ Q
    negs = []
    for j in range(Q.shape[1]):
        used, nid = {val['qcl'][j]}, []
        for r in np.argsort(-S[:, j]):
            c = val['dbcl'][r]
            if c not in used:
                nid.append(int(r))
                used.add(c)
            if len(nid) == n_neg:
                break
        negs.append(nid)
    return negs


def val_loss(X, Q, val, negs, margin):
    """Mean contrastive loss on the fixed val tuples (L2 descriptors, as in training)."""
    tot, n = 0., 0
    for j, p in enumerate(val['pos']):
        if p < 0:
            continue
        dp = np.linalg.norm(Q[:, j] - X[:, p])
        dn = np.linalg.norm(Q[:, j:j + 1] - X[:, negs[j]], axis=0)
        tot += 0.5 * dp ** 2 + 0.5 * (np.clip(margin - dn, 0, None) ** 2).sum()
        n += 1
    return tot / max(n, 1)


def val_map(X, Q, gnd):
    m, pr = compute_map(np.argsort(-(X.T @ Q), axis=0), gnd)
    return round(100 * m, 2), round(100 * pr[2], 2)


# ------------------------------------------------------------------ main
class EvalV2:
    def __init__(self, cfg, device, inputs, out):
        self.cfg, self.device, self.inputs, self.out = cfg, device, inputs, out
        self.saved_w = {v: pickle.load(open(inputs[v]['whiten'], 'rb')) for v in VERSIONS}

    def ext(self, net, paths, scales, bbxs=None, desc=''):
        return extract(net, paths, self.cfg['test_imsize'], scales, bbxs=bbxs, desc=desc, heads=True,
                       num_workers=self.cfg['num_workers'], head_list=ALL_HEADS)

    def testset(self, net, ds, scales, tag):
        g = load_gnd(self.cfg['data_root'], ds)
        d = f"{self.cfg['data_root']}/test/{ds}/jpg"
        db = self.ext(net, [f'{d}/{n}.jpg' for n in g['imlist']], scales, desc=f'{tag} {ds} db')
        q = self.ext(net, [f'{d}/{n}.jpg' for n in g['qimlist']], scales, bbxs=[x['bbx'] for x in g['gnd']],
                     desc=f'{tag} {ds} query')
        return db, q

    def whitening(self, net, scales, tag, heads):
        d = f"{self.cfg['data_root']}/train/retrieval-SfM-120k"
        db = pickle.load(open(f'{d}/retrieval-SfM-120k-whiten.pkl', 'rb'))
        qi, pi = np.array(db['qidxs']), np.array(db['pidxs'])
        sel = np.random.RandomState(0).permutation(len(qi))[:self.cfg['whiten_max_pairs']]   # same pairs as B0-B3
        qi, pi = qi[sel], pi[sel]
        used = np.unique(np.concatenate([qi, pi]))
        remap = {int(o): n for n, o in enumerate(used)}
        per = self.ext(net, [cid2filename(db['cids'][i], f'{d}/ims') for i in used], scales, desc=f'{tag} whiten')
        q2, p2 = [remap[int(i)] for i in qi], [remap[int(i)] for i in pi]
        p = net.pool.p.item()
        W = {}
        for h in heads:
            v = {'ss': get_scale(per[h], 1)}
            if len(scales) > 1:
                v['ms'] = combine_ms(per[h], ms_power(h, p))
            W[h] = {sc: whitenlearn(x.numpy().astype(np.float64), q2, p2) for sc, x in v.items()}
        pickle.dump(W, open(f'{self.out.out_dir}/whiten_{tag}.pkl', 'wb'))
        return W

    def score(self, group, stage, vecs, p, W, heads, scales=('ss', 'ms'), qe=None):
        """vecs: {ds: (db, q)} with {head: {scale: D x N}}; qe: list of (alpha, n) applied on +Lw descriptors."""
        for ds, (dbh, qh) in vecs.items():
            gnd = load_gnd(self.cfg['data_root'], ds)['gnd']
            for h in heads:
                dv, qv = variants(dbh[h], ms_power(h, p)), variants(qh[h], ms_power(h, p))
                for sc in scales:
                    for lw in [False, True]:
                        X, Q = dv[sc].numpy(), qv[sc].numpy()
                        if lw:
                            X, Q = whitenapply(X, *W[h][sc]), whitenapply(Q, *W[h][sc])
                        self.out.add_test(group, stage, ds, h, sc, lw, '-', eval_ranks(np.argsort(-(X.T @ Q), axis=0), gnd))
                        if lw and qe and h == 'cls':
                            for a, n in qe:
                                r = eval_ranks(np.argsort(-(X.T @ aqe(X, Q, n=n, alpha=a)), axis=0), gnd)
                                self.out.add_test(group, stage, ds, h, sc, lw, f'a{a}_n{n}', r)
        self.out.save()

    # ---------------------------------------------------------------- val
    def val_descriptors(self, net, val, tag, heads):
        q = self.ext(net, val['qpaths'], [1], desc=f'{tag} val q')
        x = self.ext(net, val['dbpaths'], [1], desc=f'{tag} val db')
        return {h: (x[h][1].numpy(), q[h][1].numpy()) for h in heads}

    def val_rows(self, v, desc, val, negs, W):
        for h, (X, Q) in desc.items():
            loss = float(val_loss(X, Q, val, negs, self.cfg['margin']))
            for lw in [False, True]:
                Xw, Qw = (whitenapply(X, *W[h]['ss']), whitenapply(Q, *W[h]['ss'])) if lw else (X, Q)
                m, p10 = val_map(Xw, Qw, val['gnd'])
                self.out.add('validation', model=v, head=h, trained_head=h == TRAINED_HEAD[v], scale='ss', Lw=lw,
                             val_mAP=m, val_mP10=p10, val_loss=round(loss, 4))
        self.out.save()

    def qe_grid(self, v, X, Q, val, W):
        X, Q = whitenapply(X, *W['cls']['ss']), whitenapply(Q, *W['cls']['ss'])
        m0, _ = val_map(X, Q, val['gnd'])
        self.out.add('validation_qe', model=v, alpha='-', n=0, val_mAP=m0)
        best = (m0, None)
        for a in QE_ALPHAS:
            for n in QE_NS:
                m, _ = val_map(X, aqe(X, Q, n=n, alpha=a), val['gnd'])
                self.out.add('validation_qe', model=v, alpha=a, n=n, val_mAP=m)
                if m > best[0]:
                    best = (m, (a, n))
        self.out.save()
        return best


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument('--ckpt-root', default='/kaggle/input')
    ap.add_argument('--data-root', default='/tmp/cbir_data')
    ap.add_argument('--out-dir', default='outputs/eval_v2')
    ap.add_argument('--val-queries', type=int, default=1000)
    ap.add_argument('--val-db', type=int, default=5000)
    ap.add_argument('--num-workers', type=int, default=4)
    a = ap.parse_args()

    cfg = get_config('b3', data_root=a.data_root, out_dir=a.out_dir, num_workers=a.num_workers)
    random.seed(0), np.random.seed(0), torch.manual_seed(0)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    os.makedirs(a.out_dir, exist_ok=True)
    os.makedirs(cfg['data_root'], exist_ok=True)
    print('torch', torch.__version__, '| device', device, '|', disk(os.path.dirname(os.path.abspath(a.data_root)) or '/'))
    t0 = time.time()
    inputs = find_inputs(a.ckpt_root)
    download_test(cfg['data_root'])
    download_train(cfg['data_root'])
    print(f'data ok ({(time.time() - t0) / 60:.1f} min)')
    load_backbone(cfg['arch'], 'cpu')  # warm the torch.hub cache once

    out = Collector(a.out_dir)
    E = EvalV2(cfg, device, inputs, out)
    val = build_val(cfg, a.val_queries, a.val_db)
    ms = cfg['multiscale']
    std_qe = [(3, 2)]

    # 1) frozen: val + test (all heads, MS) + Lw ----------------------------------------------------------
    net = make_net(cfg, device, 'frozen')
    vd = E.val_descriptors(net, val, 'frozen', ALL_HEADS)
    negs = mine_val_negatives(val, *vd['cls'])           # fixed val tuples, reused for every model
    Wf = E.whitening(net, ms, 'frozen', ALL_HEADS)
    E.val_rows('frozen', vd, val, negs, Wf)
    vecs_f = {ds: E.testset(net, ds, ms, 'frozen') for ds in cfg['datasets']}
    pf = net.pool.p.item()
    val_desc = {'frozen': vd['cls']}
    del net
    torch.cuda.empty_cache()

    # 2) B0 checkpoint: val + GeM ablation on test -------------------------------------------------------
    net = make_net(cfg, device, 'B0', inputs['B0']['ckpt'])
    vd = E.val_descriptors(net, val, 'B0', ALL_HEADS)
    W0 = E.whitening(net, ms, 'B0', ALL_HEADS)
    E.val_rows('B0', vd, val, negs, W0)
    vecs_0 = {ds: E.testset(net, ds, ms, 'B0') for ds in cfg['datasets']}
    p0 = net.pool.p.item()
    E.score('ablation_gem', 'B0', vecs_0, p0, W0, ['gem', 'gem_shift', 'gem_signed', 'avg', 'cls'])
    del net, vecs_0
    torch.cuda.empty_cache()
    E.score('ablation_gem', 'frozen', vecs_f, pf, Wf, ['gem', 'gem_shift', 'gem_signed', 'avg', 'cls'])

    # 3) B1-B3: val with their own (saved) whitening -----------------------------------------------------
    for v in ['B1', 'B2', 'B3']:
        net = make_net(cfg, device, v, inputs[v]['ckpt'])
        vd = E.val_descriptors(net, val, v, ['gem', 'cls', 'avg'])
        E.val_rows(v, vd, val, negs, E.saved_w[v])
        val_desc[v] = vd['cls']
        if v == 'B3':
            b3 = net
        else:
            del net
            torch.cuda.empty_cache()

    # 4) model selection on val (trained head, SS + Lw) and alpha-QE grid on val -------------------------
    rows = [r for r in out.rows['validation'] if r['trained_head'] and r['Lw']]
    sel = max(rows, key=lambda r: (r['val_mAP'], -r['val_loss']))
    print('selected on val:', sel)
    out.add('selection', **sel)
    best = {}
    for v in dict.fromkeys(['frozen', sel['model'], 'B3']):     # B3 = the model reported as B4 in v1
        if v in val_desc:
            best[v] = E.qe_grid(v, *val_desc[v], val, Wf if v == 'frozen' else E.saved_w[v])
    ref = sel['model'] if sel['model'] in best else 'B3'
    chosen = best[ref][1]                                      # None = no QE is best on val
    out.add('selection', model=f'alpha-QE (grid on {ref})', chosen_alpha=chosen[0] if chosen else '-',
            chosen_n=chosen[1] if chosen else 0, val_mAP=best[ref][0])
    out.save()
    qe_list = list(dict.fromkeys(std_qe + ([chosen] if chosen else [])))

    # 5) test: frozen baseline (same protocol) and B0-B3 re-scored from their saved descriptors -----------
    E.score('frozen', 'frozen', vecs_f, pf, Wf, ['gem', 'cls', 'avg'], qe=qe_list)
    for v in VERSIONS:
        vecs = {}
        for ds, path in inputs[v]['vecs'].items():
            s = torch.load(path, map_location='cpu')
            vecs[ds] = (s['db'], s['q'])
        E.score('test_all', v, vecs, 3.0, E.saved_w[v], ['cls'], qe=qe_list)  # p only matters for MS GeM

    # 6) B6: B3 with upscaled scale set [sqrt2, 1, 1/sqrt2] ------------------------------------------------
    W_up = E.whitening(b3, UP_SCALES, 'B3_up', ['cls'])
    for ds, path in inputs['B3']['vecs'].items():
        s = torch.load(path, map_location='cpu')
        up_db, up_q = E.testset(b3, ds, [SQ2], 'B3_up')
        gnd = load_gnd(cfg['data_root'], ds)['gnd']
        for name, scl in [('ss', [1]), ('ms', [1, 1 / SQ2, 0.5]), ('ms_up', UP_SCALES)]:
            def pick(src, up):
                return {k: (get_scale(up['cls'], k) if abs(k - SQ2) < 1e-6 else get_scale(src['cls'], k)) for k in scl}
            dbv, qv = pick(s['db'], up_db), pick(s['q'], up_q)
            X = combine_ms(dbv, 1).numpy() if len(scl) > 1 else dbv[1].numpy()
            Q = combine_ms(qv, 1).numpy() if len(scl) > 1 else qv[1].numpy()
            Wsc = W_up['cls']['ms'] if name == 'ms_up' else E.saved_w['B3']['cls'][name]
            for lw in [False, True]:
                Xw, Qw = (whitenapply(X, *Wsc), whitenapply(Q, *Wsc)) if lw else (X, Q)
                out.add_test('B6_upscale', 'B3', ds, 'cls', name, lw, '-', eval_ranks(np.argsort(-(Xw.T @ Qw), axis=0), gnd))
                if lw:
                    for al, n in qe_list:
                        r = eval_ranks(np.argsort(-(Xw.T @ aqe(Xw, Qw, n=n, alpha=al)), axis=0), gnd)
                        out.add_test('B6_upscale', 'B3', ds, 'cls', name, lw, f'a{al}_n{n}', r)
        out.save()
    print(f'done in {(time.time() - t0) / 3600:.2f} h; results in {a.out_dir}')


if __name__ == '__main__':
    main()


In [ ]:
!nvidia-smi -L
!python -u -m src.eval_v2 --ckpt-root /kaggle/input --data-root /tmp/cbir_data --out-dir /kaggle/working/eval_v2

In [ ]:
import pandas as pd, glob
for f in sorted(glob.glob('/kaggle/working/eval_v2/*/results.csv')):
    print('=====', f); df = pd.read_csv(f); print(df.to_string(max_rows=200))